In [1]:
import pandas as pd
import numpy as np
import re
import pickle
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to C:\Users\Shiva
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('IFND.csv', encoding='latin1', on_bad_lines='skip')

In [4]:
def clean_indian_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Clear web URLs
    text = re.sub(r'@\w+|\#\w+', '', text) # Clear social tags
    text = re.sub(r'[^\w\s]', '', text) # Clear punctuation
    return ' '.join([w for w in text.split() if w not in stop_words])

In [5]:
df['cleaned_text'] = df['Statement'].astype(str).apply(clean_indian_text)

In [6]:
df['Label'] = df['Label'].astype(str).str.strip()
mapping = {'TRUE': 1, 'Fake': 0}
df['Label'] = df['Label'].map(mapping)

In [7]:
df = df.dropna(subset=['cleaned_text', 'Label'])
df = df[df['cleaned_text'].str.strip() != ""]
df['Label'] = df['Label'].astype(int)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['Label'], test_size=0.2, random_state=42)

In [9]:
tfidf = TfidfVectorizer(max_df=0.7, max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [10]:
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

In [11]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42)
pac.fit(X_train_tfidf, y_train)
print(f"PAC Model Validation Accuracy: {accuracy_score(y_test, pac.predict(X_test_tfidf))*100:.2f}%")
with open('model_pac.pkl', 'wb') as f: pickle.dump(pac, f)

🎯 PAC Model Validation Accuracy: 92.69%


In [12]:
lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train_tfidf, y_train)
print(f"LR Model Validation Accuracy: {accuracy_score(y_test, lr.predict(X_test_tfidf))*100:.2f}%")
with open('model_lr.pkl', 'wb') as f: pickle.dump(lr, f)

🎯 LR Model Validation Accuracy: 93.85%


In [13]:
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
rf.fit(X_train_tfidf, y_train)
print(f"RF Model Validation Accuracy: {accuracy_score(y_test, rf.predict(X_test_tfidf))*100:.2f}%")
with open('model_rf.pkl', 'wb') as f: pickle.dump(rf, f)

🎯 RF Model Validation Accuracy: 88.48%
